In [9]:
import geopandas as gpd
from tobler.area_weighted import area_interpolate

In [10]:
# TTS data
tts = gpd.read_file("../../data/census/tts/tts2022.geojson")

# ADA wide data
ada = gpd.read_file("../../data/census/ada-wide/toronto-ada-wide.geojson")

In [11]:
# Areal interpolation

# Reproject to UTM 17N
tts = tts.to_crs(epsg=32617)
ada = ada.to_crs(epsg=32617)

# Fix invalid geometries
tts["geometry"] = tts.geometry.make_valid()
ada["geometry"] = ada.geometry.make_valid()

# Drop tts NA
tts_valid = tts.dropna(subset=["Perc_No_Veh", "Pop_Dens", "Veh_Per_Hhld"], how="all").copy()

# Interpolate
result = area_interpolate(
    source_df=tts_valid,
    target_df=ada,
    intensive_variables=["Perc_No_Veh", "Pop_Dens", "Veh_Per_Hhld"],
)

# Attach to ada
ada["perc_no_veh_interp"] = result["Perc_No_Veh"].values
ada["pop_dens_interp"] = result["Pop_Dens"].values
ada["veh_per_hhld_interp"] = result["Veh_Per_Hhld"].values

# Project to WGS84 (EPSG:4326)
ada = ada.to_crs(epsg=4326)


In [12]:
# Write files
ada.to_file(
    "../../data/census/ada-wide/ada-wide-tts.geojson", 
    driver="GeoJSON"
)

ada.to_file(
    "../../data/census/ada-wide/ada-wide-tts.gpkg",
    layer="ada",
    driver="GPKG"
)